In [6]:
import nltk, re
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
import pandas as pd
from pathlib import Path

[nltk_data] Downloading package stopwords to C:\Users\Ash
[nltk_data]     Dash\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Ash
[nltk_data]     Dash\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Ash
[nltk_data]     Dash\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
wnl = WordNetLemmatizer()
ps = PorterStemmer()

In [10]:
parent_folder = Path.cwd().parent

In [11]:
data = pd.read_csv(parent_folder.joinpath("Supporting_Docs").joinpath("tripadvisor_hotel_reviews.csv"))

In [4]:
data['astrix2star'] =data.apply(lambda x: re.sub(r'[*]',"star",x['Review']), axis = 1) 

In [5]:
data['lower'] = data.apply(lambda x : x['astrix2star'].lower(),axis = 1)

In [6]:
removewords = stopwords.words('english')

In [7]:
if 'not' in removewords:
    removewords.remove('not')

In [8]:
data['meaningful_words_only'] = data.apply(lambda x : ' '.join(words for words in x['lower'].split() if words not in removewords), axis = 1)

In [9]:
data['no_punctuations'] =data.apply(lambda x: re.sub(r'[^\w\s]',"",x['meaningful_words_only']), axis = 1) 

In [10]:
data['tokenized'] = data.apply(lambda x : word_tokenize(x['no_punctuations']),axis = 1)

In [11]:
data['Stemmed'] = data.apply(lambda x: [ps.stem(word) for word in x['tokenized']], axis = 1)

In [12]:
data['Lemmatized'] = data.apply(lambda x: [wnl.lemmatize(word) for word in x['tokenized']], axis = 1)

In [13]:
full_lemmatized = sum(data['Lemmatized'], [])
full_stemmed = sum(data['Stemmed'], [])

In [14]:
unigram_s = pd.Series(nltk.ngrams(full_stemmed,1)).value_counts()
bigram_s = pd.Series(nltk.ngrams(full_stemmed,2)).value_counts()
trigram_s = pd.Series(nltk.ngrams(full_stemmed,3)).value_counts()

unigram_l = pd.Series(nltk.ngrams(full_lemmatized,1)).value_counts()
bigram_l = pd.Series(nltk.ngrams(full_lemmatized,2)).value_counts()
trigram_l = pd.Series(nltk.ngrams(full_lemmatized,3)).value_counts()

In [15]:
unigram_l

(hotel,)           292
(room,)            275
(great,)           126
(not,)             122
(stay,)             95
                  ... 
(175,)               1
(smackagainst,)      1
(2x,)                1
(80,)                1
(connected,)         1
Name: count, Length: 2589, dtype: int64

In [16]:
bigram_l

(great, location)     24
(space, needle)       21
(hotel, monaco)       16
(great, hotel)        12
(staff, friendly)     12
                      ..
(didnt, make)          1
(personnel, didnt)     1
(minute, stay)         1
(starting, minute)     1
(food, raffle)         1
Name: count, Length: 8263, dtype: int64

In [17]:
trigram_l

(pike, place, market)            8
(view, space, needle)            5
(hotel, great, location)         5
(inn, queen, anne)               4
(room, king, bed)                4
                                ..
(hotel, dissapointment, trip)    1
(dissapointment, trip, 3)        1
(trip, 3, night)                 1
(3, night, stay)                 1
(hotel, right, street)           1
Name: count, Length: 9288, dtype: int64

In [18]:
Ngram = pd.Series(nltk.ngrams(full_lemmatized,5)).value_counts()
Ngram

(room, high, floor, great, view)                            2
(room, really, comfortable, clean, location)                2
(space, needle, experience, music, project)                 2
(loved, foodie, extemely, important, good)                  1
(fabulous, food, recommendation, city, tried)               1
                                                           ..
(agency, unfortunately, warwick, seattle, hotel)            1
(unfortunately, warwick, seattle, hotel, dissapointment)    1
(warwick, seattle, hotel, dissapointment, trip)             1
(seattle, hotel, dissapointment, trip, 3)                   1
(food, raffle, hotel, right, street)                        1
Name: count, Length: 9400, dtype: int64